# **Initialization**

In [1]:
"""Start"""

'Start'

In [14]:
#%load_ext autoreload
%reload_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import math
import random
import sys
import pulp
import vrplib
import re
import sys
import os
import gc
import glob
import contextlib
import modified_didppy as m_dp
from scipy.sparse.csgraph import minimum_spanning_tree
from scipy.optimize import linear_sum_assignment
from numpy.linalg import eigh
import time as pytime
import pulp
from ortools.linear_solver import pywraplp
from functools import lru_cache

# **Configuration & Data input**

In [15]:
# --- CONFIGURATION ---
# Path to your n20 folder containing .txt files
DATA_DIR = r"C:\Users\ACER\Desktop\Code\0.Thesis implementation\2_DIDP_custom_search_guidance_local\Thesis_modified_DIDP\2_TSPTW_dual_bounds_and_models\n20"
INPUT_CSV = "TSPTW_single_dual_bound_results.csv"
OUTPUT_CSV = "result_of_ea_dual_bounds.csv"
LOGS_DIR = "batch_logs"

if not os.path.exists(LOGS_DIR):
    os.makedirs(LOGS_DIR)

# --- GLOBAL VARIABLES (Initialize with Dummy Data) ---
# We create these so the functions in Cell 3 don't crash if checked early.
# These will be overwritten by the loop in Cell 4.
current_num_locations = 5.0
current_travel_cost = [[0]*5 for _ in range(5)]
current_avail_time = [0.0]*5
current_due_date = [1000.0]*5

print("✅ Globals initialized.")


def read_tsptw_data(file_path):
    """
    Reads TSPTW data from the specified file format:
    - Line 1: Number of locations (N)
    - Next N lines: Distance Matrix (N x N)
    - Next N lines: Time Windows (Ready Time, Due Date)
    - (Ignores subsequent lines, e.g., coordinates)
    """
    with open(file_path, 'r') as f:
        # Read all tokens (whitespace separated) to handle newlines flexibly
        tokens = f.read().split()
    
    iterator = iter(tokens)
    
    try:
        # 1. Number of locations
        num_locations = int(next(iterator))

        # 2. Travel Cost Matrix (N x N)
        travel_cost = []
        for _ in range(num_locations):
            row = []
            for _ in range(num_locations):
                row.append(float(next(iterator))) # Load as float
            travel_cost.append(row)
            
        # 3. Time Windows (N lines of: Ready_Time Due_Date)
        time_windows = []
        for _ in range(num_locations):
            ready = float(next(iterator)) # Load as float
            due = float(next(iterator))   # Load as float
            time_windows.append((ready, due))
        avail_time = [tw[0] for tw in time_windows]
        due_date = [tw[1] for tw in time_windows]
        return num_locations, travel_cost, avail_time, due_date

    except StopIteration:
        raise ValueError(f"Error reading file {file_path}: Unexpected end of file.")



✅ Globals initialized.


# **Model and dual bounds declaration**

In [16]:
def creation_of_didp_model_function():
    # Expects global variables: num_locations, dist_matrix, time_windows
    num_locations = current_num_locations
    travel_cost = current_travel_cost
    avail_time = current_avail_time
    due_date = current_due_date
    # 1. Setup Model
    model = m_dp.Model(float_cost=True)
    customer = model.add_object_type(number=num_locations)

    # 2. State Variables
    # unvisited: Set of customers to visit (excluding depot 0)
    unvisited = model.add_set_var(object_type=customer, target=list(range(1, num_locations)))
    # location: Current node
    location = model.add_element_var(object_type=customer, target=0)
    # time: Current cumulative time (resource)
    curr_time = model.add_float_resource_var(target=0.0, less_is_better=True)

    # 3. Data Tables & Helpers
    travel_time_table = model.add_float_table(travel_cost)
    
    # Separate time windows into lists for easy access

    # 4. Transitions: Visit Customer j
    for j in range(1, num_locations):
        visit = m_dp.Transition(
            name="visit {}".format(j),
            cost=travel_time_table[location, j] + m_dp.FloatExpr.state_cost(),
            preconditions=[
                unvisited.contains(j),
                # Feasibility check: Must arrive at j by its Due Date
                # Note: We can arrive early and wait, so we check if arrival <= due_date
                curr_time + travel_time_table[location, j] <= due_date[j]
            ],
            effects=[
                (unvisited, unvisited.remove(j)),
                (location, j),
                # Time update: max(arrival_time, ready_time)
                # arrival_time = curr_time + travel_time
                (curr_time, m_dp.max(curr_time + travel_time_table[location, j], avail_time[j])),
            ],
        )
        model.add_transition(visit)

    # 5. Transition: Return to Depot (0)
    return_to_depot = m_dp.Transition(
        name="return",
        cost=travel_time_table[location, 0] + m_dp.FloatExpr.state_cost(),
        effects=[
            (location, 0),
            (curr_time, curr_time + travel_time_table[location, 0]),
        ],
        preconditions=[
            unvisited.is_empty(), 
            location != 0,
        ],
    )
    model.add_transition(return_to_depot)

    # 6. Base Case
    model.add_base_case([unvisited.is_empty(), location == 0])

    for j in range(1, num_locations):
        model.add_state_constr(
            ~unvisited.contains(j) | (curr_time + travel_time_table[location, j] <= due_date[j])
        )

    # 7. Dual Bounds (Updated for Float Cost)
    # Min outgoing cost from unvisited nodes
    min_to = model.add_float_table(
        [min(travel_cost[k][j] for k in range(num_locations) if k != j) for j in range(num_locations)]
    )
    model.add_dual_bound(min_to[unvisited] + (location != 0).if_then_else(min_to[0], 0.0))

    # Min incoming cost to unvisited nodes
    min_from = model.add_float_table(
        [min(travel_cost[j][k] for k in range(num_locations) if k != j) for j in range(num_locations)]
    )
    model.add_dual_bound(
        min_from[unvisited] + (location != 0).if_then_else(min_from[location], 0.0)
    )

    # 8. Bundle
    metadata = {
        "num_locations": num_locations,
        "distance_matrix": travel_cost,
        "avail_time": avail_time,
        "due_date": due_date,
        "unvisited_var": unvisited,
        "location_var": location,
        "time_var": curr_time
    }
    
    didp_bundle = (model, metadata)
    return didp_bundle

# **Execution**

In [ ]:
# ==========================================
# 1. Configuration & File Selection
# ==========================================

# Directory containing the instances (Update this path if needed)
folder_path = r"C:\Users\ACER\Desktop\Code\0.Thesis implementation\2_DIDP_custom_search_guidance_local\Thesis_modified_DIDP\2_TSPTW_dual_bounds_and_models\n50"

# Get all .txt files
all_files = glob.glob(os.path.join(folder_path, "*.txt"))

# Select 20 random instances (or all if less than 20)
num_instances_to_test = 20
if len(all_files) > num_instances_to_test:
    selected_files = random.sample(all_files, num_instances_to_test)
else:
    selected_files = all_files

print(f"Found {len(all_files)} files. Selected {len(selected_files)} for testing.")
print("Selected Instances:")
for f in selected_files:
    print(f" - {os.path.basename(f)}")
print("-" * 50)

# ==========================================
# 2. Testing Loop
# ==========================================

results_data = []
output_csv_name = "TSPTW_single_dual_bound_50_cus_selected_results_1800s_lim.csv"

# Ensure global variables exist (initializing them)
current_num_locations = 0
current_travel_cost = []
current_avail_time = []
current_due_date = []

for i, file_path in enumerate(selected_files):
    instance_name = os.path.basename(file_path)
    print(f"\n[{i+1}/{len(selected_files)}] Processing: {instance_name}")
    
    try:
        # --- A. Read Data & Update Globals ---
        # We read data and IMMEDIATELY update the global variables that 
        # creation_of_didp_model_function relies on.
        n_loc, t_cost, a_time, d_date = read_tsptw_data(file_path)
        
        # Inject into globals
        current_num_locations = n_loc
        current_travel_cost = t_cost
        current_avail_time = a_time
        current_due_date = d_date
        
        # --- B. Initialize Model ---
        # We call your existing function, which reads the globals we just set
        didp_bundle = creation_of_didp_model_function()
        model, metadata = didp_bundle # Unpack the tuple
        
        # --- C. Solver Execution ---
        t_start = pytime.time()
        
        # Solver with 30 minute limit (1800 seconds) or similar to TSP config
        solver = m_dp.CABS(
            model,
            quiet=False, # Set to True to reduce console spam
            time_limit=1800
        )
        
        solution = solver.search()
        
        t_end = pytime.time()
        duration = t_end - t_start
        
        # --- D. Logging Results ---
        if solution.is_optimal:
             cost = solution.cost
             status = "True"
        elif solution.cost is not None:
             cost = solution.cost
             status = "False (Time Limit)"
        else:
             cost = "Inf"
             status = "False (No Sol)"

        nodes_gen = solution.generated
        nodes_exp = solution.expanded
        
        print(f"   -> Done. Cost: {cost}, Time: {duration:.2f}s, Optimal: {status}")

        # Append to results list
        results_data.append({
            "Instance": instance_name,
            "Cost": cost,
            "Nodes Expanded": nodes_exp,
            "Nodes Generated": nodes_gen,
            "Running Time (s)": duration,
            "Is Optimal": status
        })

    except Exception as e:
        print(f"   -> ERROR processing {instance_name}: {e}")
        # Optional: print full traceback if debugging
        # import traceback
        # traceback.print_exc()
        
        results_data.append({
            "Instance": instance_name,
            "Cost": "Error",
            "Nodes Expanded": 0,
            "Nodes Generated": 0,
            "Running Time (s)": 0,
            "Is Optimal": "Error"
        })

    # --- E. Intermediate Save ---
    # Save after every iteration for safety
    df_results = pd.DataFrame(results_data)
    df_results.to_csv(output_csv_name, index=False)

print("\n" + "="*50)
print("Batch Testing Complete.")
print(f"Results saved to {output_csv_name}")
print(df_results)

Found 100 files. Selected 20 for testing.
Selected Instances:
 - 85.txt
 - 12.txt
 - 14.txt
 - 72.txt
 - 46.txt
 - 3.txt
 - 64.txt
 - 56.txt
 - 84.txt
 - 27.txt
 - 10.txt
 - 31.txt
 - 59.txt
 - 80.txt
 - 75.txt
 - 0.txt
 - 29.txt
 - 51.txt
 - 52.txt
 - 48.txt
--------------------------------------------------

[1/20] Processing: 85.txt
   -> Done. Cost: 1664.0, Time: 2.59s, Optimal: True

[2/20] Processing: 12.txt
   -> Done. Cost: 1551.0, Time: 4.82s, Optimal: True

[3/20] Processing: 14.txt
   -> Done. Cost: 1723.0, Time: 1.36s, Optimal: True

[4/20] Processing: 72.txt
   -> Done. Cost: 1718.0, Time: 3.45s, Optimal: True

[5/20] Processing: 46.txt
   -> Done. Cost: 1667.0, Time: 2.24s, Optimal: True

[6/20] Processing: 3.txt
   -> Done. Cost: 1628.0, Time: 2.29s, Optimal: True

[7/20] Processing: 64.txt
   -> Done. Cost: 1798.0, Time: 1.89s, Optimal: True

[8/20] Processing: 56.txt
   -> Done. Cost: 1641.0, Time: 4.38s, Optimal: True

[9/20] Processing: 84.txt
   -> Done. Cost: 1440.